<a href="https://colab.research.google.com/github/shaikmujimel-pixel/C-Lang/blob/main/250200362_Mujimel_Week_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [12]:
import re
import time

In [2]:
url= "https://www.secondhandbazaar.in/furniture/beds-wardrobes"

In [3]:
response = requests.get(url)
response

<Response [200]>

In [4]:
response.status_code

200

In [5]:
soup = BeautifulSoup(response.content, 'html.parser')

In [6]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <link as="image" href="/logo.webp" rel="preload"/>
  <link data-precedence="next" href="/_next/static/chunks/1ttm2eq0iaqo0.css" rel="stylesheet"/>
  <link as="script" fetchpriority="low" href="/_next/static/chunks/0bc5d9o3i3s3e.js" rel="preload"/>
  <script async="" src="/_next/static/chunks/0ok9w2mn8mxa2.js">
  </script>
  <script async="" src="/_next/static/chunks/15_r4edqm-tog.js">
  </script>
  <script async="" src="/_next/static/chunks/3c-ud-775vzsp.js">
  </script>
  <script async="" src="/_next/static/chunks/turbopack-2u1d16houkiyo.js">
  </script>
  <script async="" src="/_next/static/chunks/02wzmny-ye9dv.js">
  </script>
  <script async="" src="/_next/static/chunks/395gen6b2_ijv.js">
  </script>
  <script async="" src="/_next/static/chunks/1sq7k7xu7veby.js">
  </script>
  <script async="" src="/_next/static/chunks/1krzd1pts8on1.js">
  </s

In [7]:
soup.title

<title>Beds &amp; Wardrobes in Furniture - Buy &amp; Sell Used Items | SecondHandBazaar</title>

In [8]:
links = soup.find_all('a', href=True)

product_links = []

for link in links:
    href = link['href']

    if '/beds-wardrobes/' in href and href != '/furniture/beds-wardrobes':
        if href not in product_links:
            product_links.append(href)

len(product_links)

20

In [14]:
product_data = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for link in product_links:

    if link.startswith('/'):
        product_url = "https://www.secondhandbazaar.in" + link
    else:
        product_url = link

    product_response = requests.get(
        product_url,
        headers=headers
    )

    product_soup = BeautifulSoup(
        product_response.content,
        'html.parser'
    )


    title_tag = product_soup.find('h1')

    title = title_tag.get_text(strip=True) if title_tag else None

    page_text = product_soup.get_text(' ', strip=True)

    price_match = re.search(
        r'INR\s*([\d,]+(?:\.\d+)?)',
        page_text
    )

    price = price_match.group(1) if price_match else None

    address = None

    breadcrumb = product_soup.find('a', href=re.compile(r'/beds-wardrobes'))


    if title_tag:
        details = title_tag.find_next('p')

        if details:
            details_text = details.get_text(' ', strip=True)

            address_match = re.search(
                r'^(.*?)\s+\d+\s+(?:minute|hour|day|week|month)s?\s+ago',
                details_text
            )

            if address_match:
                address = address_match.group(1)

            else:
                address = details_text.split(' ago')[0] if ' ago' in details_text else details_text


    uploaded_time = None

    time_match = re.search(
        r'\b\d+\s+(?:minute|hour|day|week|month)s?\s+ago\b',
        page_text
    )

    if time_match:
        uploaded_time = time_match.group()
    category = breadcrumb.get_text(strip=True) if breadcrumb else "Beds & Wardrobes"

    dict_product = {
        "title": title,
        "price": price,
        "address": address,
        "uploaded_time": uploaded_time,
        "category": category
    }

    product_data.append(dict_product)

    time.sleep(0.5)

In [16]:
df = pd.DataFrame(product_data)
df.head()

,title,price,address,uploaded_time,category
0,Cot + Bed,"4,600",Rs,17 hours ago,Beds & Wardrobes
1,Almira wooden,"9,999",manoj topre,21 hours ago,Beds & Wardrobes
2,Furniture,"10,000",Saurabh,1 days ago,Beds & Wardrobes
3,2-Door Wooden Wardrobe with Lockable Drawer & ...,"4,500",ADVAIT GAIKWAD,1 days ago,Beds & Wardrobes
4,King size iron cot with mattress,"6,000",sidhant kumar tiwari,1 days ago,Beds & Wardrobes


In [21]:
DB_HOST = "YOUR_POOLER_HOST"
DB_PORT = "5432"
DB_NAME = "postgres"
DB_USER = "postgres.YOUR_PROJECT_REF"
